# 0단계 코랩 스모크 테스트

목적: 본 실험 전에 **설치 · 데이터 · MuJoCo 롤아웃 · 짧은 학습**이 코랩에서 되는지, 그리고 **시간이 얼마나 걸리는지** 확인한다.

실행 방법: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고른 뒤, 위에서부터 셀을 하나씩 실행한다.
각 셀의 **마지막 출력**(특히 `[결과]`로 시작하는 줄과 오류 메시지)을 복사해서 채팅에 붙여 주면 된다.

In [ ]:
# 1. GPU와 파이썬 버전 확인
import sys, torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('[결과] python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. robosuite v1.5.1 + robomimic 소스 설치 (robomimic 공식 문서: v1.5 데이터셋은 robosuite v1.5.1 브랜치 사용)
import time; t0 = time.time()
%cd /content
!git clone -q https://github.com/ARISE-Initiative/robosuite.git
%cd /content/robosuite
!git checkout -q v1.5.1
!pip install -q -e .
%cd /content
!git clone -q https://github.com/ARISE-Initiative/robomimic.git
%cd /content/robomimic
# robomimic은 transformers==4.41.2 를 고정 요구하는데, 이 버전은 코랩 Python 3.13용 설치 파일이 없어 빌드에 실패한다(10/11 확인).
# 그래서 본체는 의존성 없이 설치하고, 필요한 패키지는 버전 고정 없이 따로 설치한다. (BC만 쓰므로 언어 모델 버전은 무관)
!pip install -q -e . --no-deps
!pip install -q psutil tqdm termcolor tensorboard tensorboardX imageio imageio-ffmpeg matplotlib egl_probe huggingface_hub transformers diffusers
%cd /content
!python -c "import robomimic, robosuite, mujoco; from robomimic.algo import algo_factory; print('[결과] import OK | robomimic', robomimic.__version__, '| robosuite', robosuite.__version__, '| mujoco', mujoco.__version__)"
print(f'[결과] 설치 {time.time()-t0:.0f}초')

In [ ]:
# 3. 데이터 다운로드 (Can MH low_dim, 107.6MB) + 공식 확인 스크립트
import os
os.environ['MUJOCO_GL'] = 'egl'   # 화면 없는 서버용 렌더링 설정
os.makedirs('/content/data', exist_ok=True)
DATA = '/content/data/can_mh_low_dim.hdf5'
if not os.path.exists(DATA):
    !wget -q -O {DATA} 'https://huggingface.co/datasets/amandlek/robomimic/resolve/main/v1.5/can/mh/low_dim_v15.hdf5?download=true'
!ls -la {DATA}
!python /content/robomimic/robomimic/scripts/get_dataset_info.py --dataset {DATA} | head -40

In [ ]:
# 4. MuJoCo 환경 생성 + 무작위 행동 롤아웃 (렌더링 없이)
import time, numpy as np
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils
ObsUtils.initialize_obs_utils_with_obs_specs({'obs': {'low_dim': ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']}})
env_meta = FileUtils.get_env_metadata_from_dataset(dataset_path=DATA)
env = EnvUtils.create_env_from_metadata(env_meta=env_meta, render=False, render_offscreen=False, use_image_obs=False)
obs = env.reset()
t0 = time.time(); steps = 400
for _ in range(steps):
    obs, r, done, info = env.step(np.random.uniform(-1, 1, 7))
dt = time.time() - t0
print('[결과] env', env.name, '| 관측 키', sorted(obs.keys())[:6], '...')
print(f'[결과] 무작위 {steps}스텝 {dt:.1f}초 ({steps/dt:.0f} 스텝/초) | 성공 판정 {env.is_success()}')

In [ ]:
# 5. BC 아주 짧게 학습 (3 epoch x 100 step) + 마지막에 롤아웃 5회 → 시간 측정
import time, torch
from robomimic.config import config_factory
from robomimic.scripts.train import train
config = config_factory(algo_name='bc')
with config.values_unlocked():
    config.experiment.name = 'smoke_bc'
    config.train.data = [{'path': DATA}]
    config.train.output_dir = '/content/runs'
    config.train.hdf5_filter_key = 'train'
    config.train.num_epochs = 3
    config.experiment.epoch_every_n_steps = 100
    config.experiment.validate = False
    config.experiment.save.enabled = False
    config.experiment.logging.log_tb = False
    config.experiment.rollout.enabled = True
    config.experiment.rollout.n = 5
    config.experiment.rollout.horizon = 400
    config.experiment.rollout.rate = 3
    config.experiment.render_video = False
    config.train.cuda = torch.cuda.is_available()
t0 = time.time()
train(config, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
print(f'[결과] 학습 3 epoch + 롤아웃 5회 총 {time.time()-t0:.0f}초')

끝. 각 셀의 `[결과]` 줄과 5번 셀 로그 중 `Rollout`/`Success_Rate`가 들어간 줄, 그리고 오류가 났다면 오류 메시지 마지막 20줄 정도를 붙여 주세요.
(3 epoch만 학습해서 성공률은 0에 가까운 게 정상이다. 이 테스트의 목적은 **돌아가는지와 시간**이다.)